In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import scipy
import scipy.spatial
import scipy.spatial.distance

def load_data(file_path):
    with open(file_path, "rb") as f:
        data = torch.load(f)
    return data.numpy()

In [2]:
X_right = load_data("X_right.trc")
Y_right = load_data("Y_right.trc")

X_left = load_data("X_left.trc")
Y_left = load_data("Y_left.trc")

X_mixed_filtered = load_data("X_mixed_filtered.trc")
Y_mixed_filtered = load_data("Y_mixed_filtered.trc")

/tmp/ipykernel_1407899/1098600258.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(f)


In [3]:
# Randomly sample 1000 points from X and got corresponding points in Y


def sample_data(X, Y, sub_data_size):
    indices = np.random.choice(X.shape[0], sub_data_size, replace=False)
    X_sub = X[indices]
    Y_sub = Y[indices]
    return X_sub, Y_sub

X_left_sub, Y_left_sub = sample_data(X_left, Y_left, 1000)
X_right_sub, Y_right_sub = sample_data(X_right, Y_right, 1000)

# combine two sub data sets and call it mixed sub data set
X_mixed_sub = np.concatenate((X_left_sub, X_right_sub), axis=0)
Y_mixed_sub = np.concatenate((Y_left_sub, Y_right_sub), axis=0)

X_mixed_filtered_sub, Y_mixed_filtered_sub = sample_data(X_mixed_filtered, Y_mixed_filtered, 1000)


# shuffle the mixed sub data set
def shuffle_data(X, Y):
    indices = np.random.permutation(X.shape[0])
    X_shuffled = X[indices]
    Y_shuffled = Y[indices]
    return X_shuffled, Y_shuffled

X_mixed_sub_shuffled, Y_mixed_sub_shuffled = shuffle_data(X_mixed_sub, Y_mixed_sub)
X_mixed_filtered_sub_shuffled, Y_mixed_filtered_sub_shuffled = shuffle_data(X_mixed_filtered_sub, Y_mixed_filtered_sub)


In [4]:
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
import ipywidgets as widgets
from IPython.display import display

def plot_clusters(Y_sub, k, num_clusters, joint_to_plot):
    # Create a slider widget for selecting the frame
    frame_slider = widgets.IntSlider(value=k, min=0, max=Y_sub.shape[1]//6 - 1, step=1, description='Frame:')

    def update_plot(k):
        Y_k = Y_sub[:, k*6:(k+1)*6]
        kmeans = KMeans(n_clusters=num_clusters, random_state=0).fit(Y_k)
        labels = kmeans.labels_

        # Count the number of points in each cluster
        counts = np.bincount(labels)

        fig = go.Figure()

        for i in range(num_clusters):
            fig.add_trace(go.Scatter3d(
                x=Y_k[labels == i, joint_to_plot[0] - 1],
                y=Y_k[labels == i, joint_to_plot[1] - 1],
                z=Y_k[labels == i, joint_to_plot[2] - 1],
                mode='markers',
                name=f'Cluster {i} (Count: {counts[i]})'
            ))

        fig.update_layout(
            scene=dict(
                xaxis_title=f'Joint {joint_to_plot[0]}',
                yaxis_title=f'Joint {joint_to_plot[1]}',
                zaxis_title=f'Joint {joint_to_plot[2]}'
            ),
            title=f'KMeans Clustering of Y Data in 3D (Frame {k})'
        )

        fig.show()

    widgets.interact(update_plot, k=frame_slider)

In [8]:
# mix data
plot_clusters(Y_mixed_sub_shuffled, 0, 2, [1, 2, 3])

interactive(children=(IntSlider(value=0, description='Frame:', max=119), Output()), _dom_classes=('widget-inte…